### Primeiro Experimento: Problema XOR

Consoante dica presente no escopo da atividade, desenvolvi completamente sozinha, antes de tudo, uma rede com uma única camada oculta e o problema XOR. Para isso, usei os conhecimentos aprendidos nas aulas e busquei vídeo aulas e materiais de estudos disponíveis na internet para compreender de maneira profunda e sincera como funciona a MLP.

Enquano desenvolvia, fui adicionando comentários que refletiam de fato aquilo que eu estava pensando. Após terminar o código e obter êxito, revisei o código novamente e adicionei demais comentários com o fito de organização, mas não retirei meus comentários informais sinceros, pois acredito que eles demonstram meu raciocínio e aprendizado com mais clareza.

In [2]:
import numpy as np

# Define a função de ativação sigmoide

def sigmoid(x):
  # sigmoid(x) = 1 / (1+e^-x)
    return 1 / (1 + np.exp(-x))

# Define a derivada da função de ativação (vou usar futuramente para calcular o gradiente)
def sigmoid_derivative(x):
    # x aqui já deve ser o valor após a aplicação da sigmóide
    return x * (1 - x)

# Dados

# Entradas
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

# Saídas esperadas
Y = np.array([[0],
              [1],
              [1],
              [0]])

# Inicialização da Rede Neural

np.random.seed(42) # Mantém os resultados reproduzíveis

input_size = 2
hidden_size = 3  # 3 neurônios na camada oculta
output_size = 1

# Inicialização aleatória de pesos e vieses
W1 = np.random.uniform(size=(input_size, hidden_size))
b1 = np.zeros((1, hidden_size))

W2 = np.random.uniform(size=(hidden_size, output_size))
b2 = np.zeros((1, output_size))

# Treinamento

epochs = 10000
learning_rate = 0.1

for epoch in range(epochs):
    # Forward Propagation
    # Camada Oculta
    Z1 = np.dot(X, W1) + b1
    A1 = sigmoid(Z1)

    # Camada de Saída
    Z2 = np.dot(A1, W2) + b2
    A2 = sigmoid(Z2) # Saída prevista pela rede

    # Calcula o Erro com MSE
    error = Y - A2

    # Exibe o erro a cada 2000 épocas
    if epoch % 2000 == 0:
        loss = np.mean(error ** 2)
        print(f"Época {epoch:05d} | Erro Quadrático Médio: {loss:.5f}")

    # Backward Propagation
    # Gradiente na camada de saída = erro * derivada da função de ativação
    d_output = error * sigmoid_derivative(A2)

    # Gradiente na camada oculta (regra da cadeia)
    error_hidden = d_output.dot(W2.T)
    d_hidden = error_hidden * sigmoid_derivative(A1)

    # Atualização dos parâmetros
    W2 += A1.T.dot(d_output) * learning_rate
    b2 += np.sum(d_output, axis=0, keepdims=True) * learning_rate

    W1 += X.T.dot(d_hidden) * learning_rate
    b1 += np.sum(d_hidden, axis=0, keepdims=True) * learning_rate

# Teste Final
print("\n Resultado Final do Teste")
# Executa um último forward pass para ver as previsões finais
Z1_final = np.dot(X, W1) + b1
A1_final = sigmoid(Z1_final)
Z2_final = np.dot(A1_final, W2) + b2
A2_final = sigmoid(Z2_final)

for i in range(len(X)):
    print(f"Entrada: {X[i]} | Saída Prevista: {A2_final[i][0]:.4f} (Esperado: {Y[i][0]})")

Época 00000 | Erro Quadrático Médio: 0.29785
Época 02000 | Erro Quadrático Médio: 0.22731
Época 04000 | Erro Quadrático Médio: 0.05918
Época 06000 | Erro Quadrático Médio: 0.00963
Época 08000 | Erro Quadrático Médio: 0.00441

 Resultado Final do Teste
Entrada: [0 0] | Saída Prevista: 0.0578 (Esperado: 0)
Entrada: [0 1] | Saída Prevista: 0.9504 (Esperado: 1)
Entrada: [1 0] | Saída Prevista: 0.9501 (Esperado: 1)
Entrada: [1 1] | Saída Prevista: 0.0519 (Esperado: 0)


### Segundo Experimento: Classificação de Dígitos (MNIST) Simples

Após já ter ganhado experiência codando uma rede neural na mão (no experimento supracitado), quis desenvolver a atividade considerando apenas os requisitos mínimos de entrega.

Assim como no experimento anterior, enquano desenvolvia, fui adicionando comentários que refletiam de fato aquilo que eu estava pensando. Paralelamente, haja vista a complexidade do código, precisei em alguns momentos interromper meu raciocínio para relembrar alguns conceitos e estruturas, e também adicionei comentários com característica mais técnica para registrar o que aprendi. Após terminar o código e obter êxito, revisei o código novamente e adicionei demais comentários com o fito de organização, mas não retirei meus comentários informais sinceros, pois acredito que eles demonstram meu raciocínio e aprendizado com mais clareza.

In [1]:
import numpy as np
from tensorflow.keras.datasets import mnist

# define a função de ativação ReLu (a função de ativação serve para atribuir não linearidade)
def relu(Z):
    # f(Z) = max(0, Z)
    # A função ReLu compara cada elemento da matriz Z com 0 e mantém apenas os valores positivos
    # 0 se Z<=0 e Z se Z>0
    return np.maximum(0, Z)

# define a derivada da função de ativação ReLu (usada para descobrir a taxa de erro no futuro)
def relu_derivative(Z):
    return Z > 0

# Define o softmax
# Usado em problemas de classificação
# Transforma as pontuações que o modelo dá em probabilidade (normaliza entre 0 e 1)
# Faz isso aplicando e^x em cada pontuação e, depois, cada valor é dividido pela soma de todos os valores exponenciados (garantindo que o resultado esteja entre 0 e 1)
def softmax(Z):
  # Vale mencionar que axis=0 diz para realizar a operação na vertical e keepdims = True garante que a matriz rsultante preserve a bidimencionalidade
    exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

# Aplica One-Hot-Encoding
# Transforma variáveis categóricas em uma nova coluna e atibui o valor 1 (para presença) ou 0 (para ausência) para cada linha do dataset.
def to_one_hot(Y, num_classes=10):
    one_hot = np.zeros((num_classes, Y.size))
    one_hot[Y, np.arange(Y.size)] = 1
    return one_hot

# Estrutura do Multilayer Perceptron
class MLP:
    def __init__(self, input_size=784, hidden_size=128, output_size=10):
        # Inicialização He / Kaiming para os pesos da ReLU, e inicialização pequena para Softmax
        # A Inicialização He / Kaiming tem como premissa que a melhor forma para manter a variável das ativações estável quando usamos a função de ativação ReLu é multiplicando os pesos por raiz quadrada de 2/nmin.
        # Isso porque, se inicializarmos pesos com valores idênticos, faremos com que todos os neurônios aprendam a mesma coisa.
        # Paralelamente, se inicializarmos os pesos com valores muito grandes ou muito pequenos, quebraremos o sinal elétrico ao longo das camadas.

        self.W1 = np.random.randn(hidden_size, input_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros((hidden_size, 1))

        self.W2 = np.random.randn(output_size, hidden_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros((output_size, 1))

    # Define o forward propagation

    def forward(self, X):

      # Z1 = W1 * X + b1
      # A1 = ReLu(Z1)
      # Z2 = W2 * A1 + b2
      # A2 = softmax(Z2)

      # Camada Oculta
        self.Z1 = np.dot(self.W1, X) + self.b1
        self.A1 = relu(self.Z1)

      # Camada de saída
        self.Z2 = np.dot(self.W2, self.A1) + self.b2
        self.A2 = softmax(self.Z2)

        # Retorna a saída softmax (probabilidade)
        return self.A2

    # Define o backward propagation
    def backward(self, X, Y_one_hot):
      # m é a quantidade de dados processados simultâneamente
      # inicializando para usar m no fututo para dividir a soma dos gradientes por m e, assim, obter a média do gradiente
        m = X.shape[1]

        # Erro Bruto da Saída = Probabilidade Prevista - Realidade Esperada
        dZ2 = self.A2 - Y_one_hot
        # Gradiente da cama de saída (Softmax + Cross-Entropy)
        self.dW2 = (1 / m) * np.dot(dZ2, self.A1.T)
        self.db2 = (1 / m) * np.sum(dZ2, axis=1, keepdims=True)

        # Gradiente da camada oculta
        dZ1 = np.dot(self.W2.T, dZ2) * relu_derivative(self.Z1)
        self.dW1 = (1 / m) * np.dot(dZ1, X.T)
        self.db1 = (1 / m) * np.sum(dZ1, axis=1, keepdims=True)

    # Atualiza os pesos e vieses
    # Para isso, multiplicamos os gradientes pela taxa de aprendizado (lr) e subtraímos dos parâmetros atuais
    def update_params(self, lr):
        self.W1 -= lr * self.dW1
        self.b1 -= lr * self.db1
        self.W2 -= lr * self.dW2
        self.b2 -= lr * self.db2

# Carregamento e Pré Processamento

print("Carregando o dataset MNIST...")
(X_train_raw, Y_train_raw), (X_test_raw, Y_test_raw) = mnist.load_data()

# Redimensionar (60000, 28, 28) para (784, 60000) e normalizar para [0, 1]
X_train = X_train_raw.reshape(X_train_raw.shape[0], -1).T / 255.0
X_test = X_test_raw.reshape(X_test_raw.shape[0], -1).T / 255.0

# Preparar os labels
Y_train_one_hot = to_one_hot(Y_train_raw)

# Treinamento

# Uma época configura-se como a passagem completa de todo o conjunto de dados de treino pela rede
epochs = 20
batch_size = 64
# A taxa de aprendizado controla o tamanho do passo, ou seja, a descida do gradiente
# Um learning_rate de 0.1 faz com que, por exemplo, se o gradiente dizer para dar um passo de 3,56 para baixo, tranformamos esse passo em 0,356 (evita, assim, ultrapassar o que queremos).
learning_rate = 0.1
num_train_samples = X_train.shape[1]

mlp = MLP(input_size=784, hidden_size=128, output_size=10)

print("\nIniciando o Treinamento...")
for epoch in range(epochs):
    # Shuffling (Embaralhar os dados a cada época)
    permutation = np.random.permutation(num_train_samples)
    X_train_shuffled = X_train[:, permutation]
    Y_train_shuffled_one_hot = Y_train_one_hot[:, permutation]

    for i in range(0, num_train_samples, batch_size):
        # Garante que não vai estourar o limite do array
        end = min(i + batch_size, num_train_samples)

        X_batch = X_train_shuffled[:, i:end]
        Y_batch = Y_train_shuffled_one_hot[:, i:end]

        # Passo de treino
        mlp.forward(X_batch)
        mlp.backward(X_batch, Y_batch)
        mlp.update_params(learning_rate)

    # Avaliação ao fim da época
    predictions_train = mlp.forward(X_train)
    acc_train = np.sum(np.argmax(predictions_train, axis=0) == Y_train_raw) / num_train_samples * 100

    print(f"Época {epoch+1:02d}/{epochs} | Acurácia no Treino: {acc_train:.2f}%")

# Teste Final
predictions_test = mlp.forward(X_test)
acc_test = np.sum(np.argmax(predictions_test, axis=0) == Y_test_raw) / X_test.shape[1] * 100
print(f"\n[Resultado Final] Acurácia no Dataset de Teste: {acc_test:.2f}%")

Carregando o dataset MNIST...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Iniciando o Treinamento...
Época 01/20 | Acurácia no Treino: 93.02%
Época 02/20 | Acurácia no Treino: 95.37%
Época 03/20 | Acurácia no Treino: 96.06%
Época 04/20 | Acurácia no Treino: 97.12%
Época 05/20 | Acurácia no Treino: 97.45%
Época 06/20 | Acurácia no Treino: 97.67%
Época 07/20 | Acurácia no Treino: 97.89%
Época 08/20 | Acurácia no Treino: 97.97%
Época 09/20 | Acurácia no Treino: 98.23%
Época 10/20 | Acurácia no Treino: 98.71%
Época 11/20 | Acurácia no Treino: 98.85%
Época 12/20 | Acurácia no Treino: 98.91%
Época 13/20 | Acurácia no Treino: 99.03%
Época 14/20 | Acurácia no Treino: 99.10%
Época 15/20 | Acurácia no Treino: 99.20%
Época 16/20 | Acurácia no Treino: 99.36%
Época 17/20 | Acurácia no Treino: 99.41%
Época 18/20 | Acurácia no Treino: 99.40%
Época 19/20 | Acurácia no Treino: 99.48%
Época 20/20 | Acurácia no Treino: 99.52%

[Resultado Final] Acurácia no Dataset de Teste: 97.88%
